# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meer140/flyrank-ml-internship-project/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This notebook builds the complete feature vector for **Lane 2**, audits missingness, and conducts a deliberate feature-leakage experiment.

## 1. Build the feature vector

We construct numeric and categorical feature matrices from observed search and engagement signals, applying log-transforms to skewed volume features.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Numeric features
numeric_cols = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'avg_position', 'ctr', 'word_count', 'content_age_days', 'days_since_last_update',
    'engagement_rate', 'scroll_rate', 'search_volume', 'cpc'
]

X_num = df[numeric_cols].fillna(0)
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'search_volume']:
    X_num[f'log_{col}'] = np.log1p(X_num[col])

# Categorical features
cat_cols = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'position_tier']
X_cat = pd.get_dummies(df[cat_cols].fillna('unknown'), drop_first=True)

X_honest = pd.concat([X_num, X_cat], axis=1)
y = (df['trend_direction'] == 'down').astype(int)

print(f'Feature vector built: {X_honest.shape[1]} features across {X_honest.shape[0]:,} rows.')

Feature vector built: 33 features across 30,000 rows.


## 2. Feature notes (meaning, missing, categorical, available-when?)

- `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`: GSC search performance signals (knowable before decision moment).
- `sessions_90d`, `engagement_rate`, `scroll_rate`: GA4 user behavior signals (knowable before decision moment).
- `word_count`, `content_age_days`, `days_since_last_update`: Content metadata signals (knowable before decision moment).
- `search_volume`, `cpc`, `competition_level`: Keyword market context signals.
- Missing values are zero-filled or assigned 'unknown' category tiers.

In [2]:
missing_summary = df[numeric_cols].isnull().sum()
print('Numeric Feature Missing Values:')
print(missing_summary[missing_summary > 0])

Numeric Feature Missing Values:
word_count       7699
scroll_rate       125
search_volume    2468
cpc              2468
dtype: int64


## 3. The leakage hunt

**Audit of Leakage Risks:**
1. `trend_direction` & `trend_pct`: Direct label derivations. MUST BE EXCLUDED from features.
2. `impressions_last_30d`: Future outcome metric. MUST BE EXCLUDED.
3. Product flags (`health_score`, priority flags): Derived composite scores. EXCLUDED to prevent circular learning.

**Deliberate Trap Experiment:** Injecting a label-derived proxy field produces artificially inflated accuracy (~98%), whereas the honest feature vector achieves ~70% AUC.

In [3]:
# Deliberate Leakage Trap Experiment
# Honest Split
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.2, random_state=42, stratify=y)
rf_h = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
rf_h.fit(X_tr, y_tr)
acc_h = accuracy_score(y_te, rf_h.predict(X_te))

# Leaked Split (Injecting trend_pct or exact proxy)
X_leaked = X_honest.copy()
X_leaked['leaked_trend_pct'] = df['trend_pct'].fillna(0)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.2, random_state=42, stratify=y)
rf_l = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
rf_l.fit(X_tr_l, y_tr)
acc_l = accuracy_score(y_te, rf_l.predict(X_te_l))

print(f'Honest Model Accuracy: {acc_h:.4f}')
print(f'Leaked Model Accuracy (TRAP TRIGGERED): {acc_l:.4f}')

Honest Model Accuracy: 0.6797
Leaked Model Accuracy (TRAP TRIGGERED): 0.9997


## 4. What I excluded and why

- `trend_direction`: The target label itself.
- `trend_pct`: Direct mathematical computation of impression drop across the target window (leaks target).
- `impressions_last_30d` / `clicks_last_30d`: Target window performance facts.
- `content_id` & `client_id`: Pseudonymized string identifiers (used for grouping/splits only, never features).
- `health_score` & product flags: Proprietary hand-tuned composite flags.

In [4]:
excluded_fields = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'health_score', 'client_id', 'content_id']
print('Explicitly Excluded Fields:')
for f in excluded_fields:
    print(f'  - {f}')

Explicitly Excluded Fields:
  - trend_direction
  - trend_pct
  - impressions_last_30d
  - health_score
  - client_id
  - content_id


## Self-check

- [x] Every section filled
- [x] Runs top to bottom
- [x] No client names/URLs
- [x] Careful words: observed, measured, directional
- [x] Committed under `work/notebooks/`